# 04 — Retrain on ATS Dataset (6,374 samples)

**Project:** TalentMatch.ai — Iteration 2  
**Goal:** Retrain DistilBERT regression model on `0xnbk/resume-ats-score-v1-en` which has 6,374 samples — nearly 10x more than our Phase 2 dataset. This should significantly reduce MAE and produce a model that generalizes better to real-world inputs.

## What changed vs notebook 02
- New dataset: 5,099 train + 1,275 validation (pre-split by the dataset authors)
- Schema: resume and JD are combined in one `text` field separated by ` SEP `
- Labels: `ats_score` is already on 0-100 scale — no dynamic normalization needed
- Target: RMSE < 8.0 (dataset authors report this is achievable)

## Notebook Flow
1. Install dependencies + authenticate
2. Load dataset and inspect schema
3. Parse text field into resume + JD
4. Analyze score distribution
5. Build PyTorch Dataset
6. Load DistilBERT with regression head
7. Train with Trainer API
8. Evaluate on validation set + compare vs Phase 2
9. Push improved model to HuggingFace Hub
10. Inference sanity check


In [ ]:
# =============================================================
# Step 1 — Install dependencies and authenticate
# =============================================================
# REMINDER: Before running this cell, set up your HF_TOKEN in
# Colab Secrets (key icon in left sidebar) with write access.
# Runtime must be set to T4 GPU.
# =============================================================

!pip install -q -U transformers accelerate huggingface_hub evaluate

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Authenticated with HuggingFace Hub.")

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================
# Step 2 — Load dataset and inspect schema
# =============================================================
# This dataset already has train/validation splits — no manual
# splitting needed. The dataset authors did this for us.
#
# Schema (confirmed from exploration):
#   text:           resume + ' SEP ' + job_description (one string)
#   ats_score:      float 0-100 (already scaled, no normalization needed)
#   original_label: string ('Good Fit', 'Potential Fit', 'No Fit')
# =============================================================

from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("0xnbk/resume-ats-score-v1-en")
print(dataset)

# Inspect one sample
sample = dataset['train'][0]
print(f"\nFields: {list(sample.keys())}")
print(f"ATS score: {sample['ats_score']}")
print(f"Label: {sample['original_label']}")
print(f"Text length: {len(sample['text'])} chars")
print(f"\nFirst 300 chars of text:")
print(sample['text'][:300])

In [ ]:
# =============================================================
# Step 3 — Parse text field into resume + JD
# =============================================================
# The text field uses ' SEP ' (with spaces on both sides) as
# the separator between resume and JD.
#
# We split on ' SEP ' and take:
#   parts[0] = resume text
#   parts[1] = job description text
#
# If a sample has no ' SEP ' separator we skip it — it's malformed.
# If it has more than one ' SEP ' we join everything after the first
# as the JD, since some resumes may contain 'SEP' in content.
# =============================================================

def parse_sample(sample):
    """Split text into resume and JD. Returns None if malformed."""
    text = sample['text']
    if ' SEP ' not in text:
        return None
    parts = text.split(' SEP ', 1)  # split on FIRST occurrence only
    resume = parts[0].strip()
    jd = parts[1].strip()
    if not resume or not jd:
        return None
    return {
        'resume': resume,
        'jd': jd,
        'score': float(sample['ats_score']),
        'label': sample['original_label'],
    }

# Parse train split
train_records, train_skipped = [], 0
for s in dataset['train']:
    parsed = parse_sample(s)
    if parsed is None:
        train_skipped += 1
    else:
        train_records.append(parsed)

# Parse validation split
val_records, val_skipped = [], 0
for s in dataset['validation']:
    parsed = parse_sample(s)
    if parsed is None:
        val_skipped += 1
    else:
        val_records.append(parsed)

train_df = pd.DataFrame(train_records)
val_df   = pd.DataFrame(val_records)

print(f"Train:      {len(train_df)} valid | {train_skipped} skipped")
print(f"Validation: {len(val_df)} valid | {val_skipped} skipped")
print(f"\nTrain score stats:")
print(train_df['score'].describe())

In [ ]:
# =============================================================
# Step 4 — Analyze score distribution
# =============================================================
# We want to confirm the distribution is reasonable before training.
# Also check label breakdown — if the dataset is heavily skewed
# toward one class, we may need to account for that.
# =============================================================

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Score distribution
sns.histplot(train_df['score'], bins=20, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('ATS Score Distribution (Train)')
axes[0].set_xlabel('Score (0-100)')

# Label breakdown
train_df['label'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Label Distribution (Train)')
axes[1].set_xlabel('Label')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("\nLabel breakdown:")
print(train_df['label'].value_counts())
print(f"\nScore range: {train_df['score'].min():.1f} - {train_df['score'].max():.1f}")
print(f"Mean: {train_df['score'].mean():.1f} | Std: {train_df['score'].std():.1f}")

In [ ]:
# =============================================================
# Step 5 — Build PyTorch Dataset
# =============================================================
# Same tokenization strategy as notebook 02:
#   [CLS] resume [SEP] jd [SEP] — max 512 tokens
#
# Label normalization:
# Even though scores are already 0-100, we still divide by 100
# during training for gradient stability (MSELoss on 0-1 vs 0-100
# makes a big difference in loss magnitude and gradient updates).
# We rescale back to 0-100 for evaluation.
# =============================================================

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ResumeJDDataset(Dataset):
    """PyTorch Dataset for resume-JD pairs with normalized labels."""
    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['resume'],
            row['jd'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(row['score'] / 100.0, dtype=torch.float)
        }

train_dataset = ResumeJDDataset(train_df, tokenizer)
val_dataset   = ResumeJDDataset(val_df,   tokenizer)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")

# Sanity check one sample
sample = train_dataset[0]
print(f"\ninput_ids shape:      {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"label (normalized):   {sample['labels'].item():.4f}")
print(f"label (0-100 scale):  {sample['labels'].item() * 100:.2f}")

In [ ]:
# =============================================================
# Step 6 — Load DistilBERT with regression head
# =============================================================
# Same architecture as notebook 02.
# num_labels=1 → MSELoss regression.
# All 66.9M parameters are trainable (full fine-tuning).
# =============================================================

import numpy as np
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


def compute_metrics(eval_pred):
    """MAE and RMSE on 0-100 scale."""
    predictions, labels = eval_pred
    predictions = predictions.flatten() * 100
    labels      = labels.flatten() * 100
    mae  = np.mean(np.abs(predictions - labels))
    rmse = np.sqrt(np.mean((predictions - labels) ** 2))
    return {
        'mae':  round(float(mae),  4),
        'rmse': round(float(rmse), 4),
    }

print("Model and metrics ready.")

In [ ]:
# =============================================================
# Step 7 — Configure TrainingArguments and train
# =============================================================
# Key difference from notebook 02:
#   - More data (5,099 vs 680) means more steps per epoch
#   - We use eval_strategy='epoch' so we see val MAE after each epoch
#   - warmup_steps=100 (more warmup since more steps per epoch)
#   - Expected training time: ~15-20 min on T4 for 4 epochs
#
# Checkpoints are saved to Drive so training isn't lost if
# Colab disconnects.
# =============================================================

from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/content/drive/MyDrive/TalentMatch-AI/checkpoints/retrain_ats"

# Mount Drive first
from google.colab import drive
drive.mount('/content/drive')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='mae',
    greater_is_better=False,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting retraining on ATS dataset...")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Steps per epoch: {len(train_dataset) // training_args.per_device_train_batch_size}")
print("---")

train_result = trainer.train()

print("\nTraining complete!")
print(f"Total steps:   {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")

In [ ]:
# =============================================================
# Step 8 — Evaluate and compare vs Phase 2
# =============================================================
# We evaluate on the validation set (no separate test set in this
# dataset — the authors pre-split it as train/validation).
#
# Phase 2 results for comparison:
#   MAE:  11.95 | RMSE: 14.90 | Dataset: 680 samples
#   Target: MAE < 10, RMSE < 8.0
# =============================================================

print("Evaluating on validation set...")
val_results = trainer.evaluate(eval_dataset=val_dataset)

# Naive baseline
mean_train_score = train_df['score'].mean()
baseline_mae  = np.mean(np.abs(val_df['score'] - mean_train_score))
baseline_rmse = np.sqrt(np.mean((val_df['score'] - mean_train_score) ** 2))

print("\n===== BENCHMARK COMPARISON =====")
print(f"{'Metric':<20} {'Baseline':>10} {'Phase 2':>10} {'This run':>10}")
print("-" * 52)
print(f"{'MAE (↓ better)':<20} {baseline_mae:>10.2f} {'11.95':>10} {val_results['eval_mae']:>10.4f}")
print(f"{'RMSE (↓ better)':<20} {baseline_rmse:>10.2f} {'14.90':>10} {val_results['eval_rmse']:>10.4f}")
print(f"{'Training samples':<20} {'N/A':>10} {'680':>10} {len(train_dataset):>10}")
print("=" * 52)

mae_improvement = 11.95 - val_results['eval_mae']
direction = 'better' if mae_improvement > 0 else 'worse'
print(f"\nThis model is {abs(mae_improvement):.2f} MAE points {direction} than Phase 2.")

In [ ]:
# =============================================================
# Step 9 — Push improved model to HuggingFace Hub
# =============================================================
# We push as a new model repo: TalentMatch-AI-v2
# This keeps the Phase 2 model intact for comparison.
# Token is loaded securely from Colab Secrets.
# =============================================================

HF_REPO = "LucasLisboadev/TalentMatch-AI-v2"

print(f"Pushing model to: https://huggingface.co/{HF_REPO}")

model.save_pretrained("/content/talentmatch_v2")
tokenizer.save_pretrained("/content/talentmatch_v2")

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"\nModel pushed successfully!")
print(f"View at: https://huggingface.co/{HF_REPO}")

In [ ]:
# =============================================================
# Step 10 — Inference sanity check
# =============================================================
# We test the same 3 pairs every time for consistency:
#   1. Strong match  — Python engineer → Python backend role
#   2. Moderate match — Marketing manager → Sales director role
#   3. Weak match    — Nurse → Python backend role
#
# Expected: strong (high) > moderate (mid) > weak (low)
# If this ordering holds, the model has learned meaningful signal.
# =============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

def predict_score(resume, jd, tokenizer, model):
    """Run inference on a resume-JD pair. Returns score 0-100."""
    inputs = tokenizer(
        resume, jd,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        output = model(**inputs)
    score = output.logits.squeeze().item() * 100
    return round(float(np.clip(score, 0, 100)), 2)

inf_tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
inf_model     = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
inf_model.eval()

# Test pair 1 — strong match
strong_resume = """Senior Backend Engineer with 6 years of experience building scalable
REST APIs using Python and FastAPI. Proficient in PostgreSQL, Redis, and Docker.
Led a team of 4 engineers to deliver a microservices architecture serving 2M daily users.
Strong experience with CI/CD pipelines, AWS deployments, and agile development practices."""
strong_jd = """We are looking for a Backend Engineer with 4+ years of Python experience.
Hands-on expertise with FastAPI or Django, PostgreSQL, and Docker required.
Experience leading small engineering teams is a strong plus."""

# Test pair 2 — moderate match
moderate_resume = """Marketing Manager with 5 years experience in B2B demand generation,
content strategy, and CRM tools including Salesforce and HubSpot. Led campaigns that
generated $2M in pipeline. Strong analytical and communication skills."""
moderate_jd = """Sales Director: 5+ years in B2B sales or marketing required.
Experience with Salesforce CRM, pipeline management, and executive presentations.
Leadership experience preferred."""

# Test pair 3 — weak match
weak_resume = """Registered Nurse with 8 years of ICU experience. Skilled in patient care,
medication administration, and clinical documentation. Epic EHR certified. ACLS/BLS certified."""
weak_jd = """We are looking for a Backend Engineer with 4+ years of Python experience.
Hands-on expertise with FastAPI or Django, PostgreSQL, and Docker required."""

strong_score   = predict_score(strong_resume,   strong_jd,   inf_tokenizer, inf_model)
moderate_score = predict_score(moderate_resume, moderate_jd, inf_tokenizer, inf_model)
weak_score     = predict_score(weak_resume,     weak_jd,     inf_tokenizer, inf_model)

print("===== INFERENCE SANITY CHECK (v2) =====")
print(f"Strong match:   {strong_score}/100")
print(f"Moderate match: {moderate_score}/100")
print(f"Weak match:     {weak_score}/100")
print()

if strong_score > moderate_score > weak_score:
    print("✓ Perfect ordering — model correctly ranks all three pairs.")
elif strong_score > weak_score:
    print("~ Partial ordering — strong > weak but moderate is off.")
else:
    print("✗ Ordering failed — model still needs improvement.")

## Summary

| Metric | Phase 2 (680 samples) | Phase 4 (5,099 samples) |
|---|---|---|
| Val MAE | 11.95 | TBD |
| Val RMSE | 14.90 | TBD |
| HF Hub | TalentMatch-AI-full | TalentMatch-AI-v2 |

*(Update after running)*

**Next step:** Open `05_domain_classifier.ipynb` to train a domain pre-filter that improves cross-domain accuracy.
